# Experimento MAREA — Variante: **MAREA-base con gamma reescalado**

Este notebook produce **el arma que falta del 2x2**.

|              | ruido absoluto | ruido escalado |
|--------------|----------------|----------------|
| **base**     | `MAREA`        | `MAREA_REL` <- **esto** |
| **bio**      | `MAREABIO`     | `MAREABIO_REL` |

### Por que hace falta

`gamma_max = 0.2` multiplica xi ~ U(-1,1) en unidades ABSOLUTAS y se calibro en
dominios de ancho ~10.24 (Sphere/Rastrigin), donde 0.2 es el 2% del ancho. En
CEC el dominio es [-100,100]: ese mismo 0.2 pasa a ser el 0.1% y el termino de
diversificacion queda inerte. Es el mismo defecto que sigma1/sigma2 en
MAREA-bio, que al corregirse llevo a MAREABIO_REL de 6o a 3o de 7 en D50.

Sin este arma, comparar `MAREABIO_REL` contra `MAREA` mezcla dos cambios
—la formulacion biologica y la escala del ruido— y le atribuye todo al primero.
La ultima celda separa los dos efectos.

### Protocolo
- 71 funciones CEC 2014 (30) + CEC 2017 (29) + CEC 2022 (12)
- 10D y 50D (CEC2022: 10D y 20D) | 51 corridas | guardado incremental
- **Una sola** configuracion de parametros: gamma_rel es un multiplicador
  constante (200/10.24 = 19.53x) igual para las 71 funciones, no un ajuste
  por problema ni por MaxFES.


In [ ]:
import sys, os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PROJECT_DIR = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from experimento_config import (
    run_experiment, load_partial, CEC_REGISTRY,
    MAXFES, DIMS, N_RUNS, POP_SIZE, RESULTS_DIR
)

ALG_NAME  = 'MAREA_REL'
N_WORKERS = 8       # el Mac mini tiene 24 cores; dejar aire si hay otra corrida
RUN_HEAVY = False   # 50k / 500k / 5M solo con True

from marea_benchmark import marea, MAREAParams, CAL_WIDTH
p = MAREAParams()
p.gamma_rel = True

assert p.early_stop is False, 'early_stop debe estar desactivado para el benchmark'
assert p.gamma_rel is True,   'este notebook es el arma con gamma reescalado'

def alg_fn(f, bounds, N, T, seed):
    p.N = N; p.T = T
    return marea(f, bounds, p, seed=seed)

print(f'\u2713 {ALG_NAME} cargado correctamente')
print(f'  gamma_rel  = {p.gamma_rel}  (escala = 200/{CAL_WIDTH} = {200/CAL_WIDTH:.2f}x)')
print(f'  early_stop = {p.early_stop}  (agota MaxFES completo)')
print(f'  Resultados en: {RESULTS_DIR}/')
print(f'  Workers: {N_WORKERS} | RUN_HEAVY: {RUN_HEAVY}')


## 1. Verificacion de entorno — **no saltarse**

Solo si dice **ESTE ES EL ENTORNO CORRECTO** los resultados seran comparables
con el resto del benchmark. Se lanza con `sys.executable`, el interprete real
del kernel: cambiar el kernel en el menu no reinicia el proceso.


In [ ]:
!{sys.executable} verificar_entorno.py


## 2. Smoke test + auditoria de presupuesto

1. **Agota MaxFES** — ninguna corrida puede parar antes de T iteraciones.
2. **No se pasa** — MAREA-base gasta 1.02x, como PSO/DE/GWO/WOA/ABC. El
   reescalado de gamma no cambia el numero de evaluaciones, solo su tamano.
3. **gamma_rel esta activo** — se comprueba que el resultado difiere del de
   `gamma_rel=False`, para no correr por error el arma que ya tienes.


In [ ]:
from experimento_config import get_problems_for_dim
import time

problems_10d = get_problems_for_dim(10)
test_prob    = problems_10d[0]
print(f'Probando con: {test_prob["name"]} | dim={test_prob["dim"]}')

T_TEST = 62   # 62 x 80 = 4,960 FES ~ MaxFES=5,000

t0 = time.time()
for seed in range(3):
    r = alg_fn(test_prob['f'], test_prob['bounds'], N=POP_SIZE, T=T_TEST, seed=seed)
    print(f'  seed={seed}  best_f={r["best_f"]:.4e}  iters={r["iters"]}/{T_TEST}')
    assert r['iters'] == T_TEST, 'Alguna corrida paro antes de agotar el presupuesto'
print(f'Tiempo: {time.time()-t0:.2f}s')

class _Contador:
    def __init__(self, f): self.f, self.n = f, 0
    def __call__(self, x):
        self.n += 1
        return self.f(x)

cnt = _Contador(test_prob['f'])
alg_fn(cnt, test_prob['bounds'], N=POP_SIZE, T=T_TEST, seed=0)
nominal = POP_SIZE * T_TEST
print(f'FES reales = {cnt.n:,}   nominal = {nominal:,}   ratio = {cnt.n/nominal:.2f}x')
assert cnt.n <= nominal + POP_SIZE, 'se pasa del presupuesto asignado'

# gamma_rel realmente activo?
q = MAREAParams(); q.N = POP_SIZE; q.T = T_TEST; q.gamma_rel = False
sin = marea(test_prob['f'], test_prob['bounds'], q, seed=0)['best_f']
con = alg_fn(test_prob['f'], test_prob['bounds'], N=POP_SIZE, T=T_TEST, seed=0)
print(f'gamma absoluto = {sin:.4e}   gamma escalado = {con["best_f"]:.4e}')
assert sin != con['best_f'], 'gamma_rel no esta teniendo efecto'
print('\u2713 Listo para el experimento.')


## Experimento — MaxFES = 5,000  ← **ejecutar esta**
*Caso de uso: optimizacion costosa / tiempo-real*

Coste: ~6 min con 8 workers. Es la unica celda que hace falta para cerrar
el 2x2. Escribe en `MAREA_REL_D{10,50}_FES5000.json` — no pisa nada.


In [ ]:
# ── MaxFES = 5,000 ──────────────────────────────
for ndim in DIMS:
    run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=5000, n_workers=N_WORKERS)


## Experimento — MaxFES = 50,000

Coste: ~17 min (D=10) + ~40 min (D=50) con 8 workers. Requiere `RUN_HEAVY = True`.


In [ ]:
if RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=50000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_HEAVY = False). Coste: ~17 min (D=10) + ~40 min (D=50) con 8 workers')


## Experimento — MaxFES = 500,000

Coste: ~2.8 h (D=10) + ~6.7 h (D=50) con 8 workers. Requiere `RUN_HEAVY = True`.


In [ ]:
if RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=500000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_HEAVY = False). Coste: ~2.8 h (D=10) + ~6.7 h (D=50) con 8 workers')


## Experimento — MaxFES = 5,000,000

Coste: ~10x el de 500k — noche completa de maquina. Requiere `RUN_HEAVY = True`.


In [ ]:
if RUN_HEAVY:
    for ndim in DIMS:
        run_experiment(ALG_NAME, alg_fn, ndim=ndim, maxfes=5000000, n_workers=N_WORKERS)
else:
    print('Saltada (RUN_HEAVY = False). Coste: ~10x el de 500k — noche completa de maquina')


## Resumen de resultados


In [ ]:
print(f'\n=== RESUMEN {ALG_NAME} ===')
print(f'{"Config":<30} {"Completadas":>12} {"Mean":>14} {"Best":>14}')
print('-' * 72)
for mf in MAXFES:
    for nd in DIMS:
        vals = list(load_partial(ALG_NAME, nd, mf).values())
        label = f'D={nd:2d} MaxFES={mf:>10,}'
        if vals:
            print(f'{label:<30} {len(vals):>12} {np.mean(vals):>14.4e} {np.min(vals):>14.4e}')
        else:
            print(f'{label:<30}   (sin resultados aun)')


## El 2x2 — cuanto es la escala y cuanto es la biologia

Tres numeros que responden la pregunta del paper:

- **efecto escala** = `MAREA_REL` / `MAREA` — cuanto mejora solo por arreglar
  las unidades del ruido, sin tocar la formulacion.
- **efecto formulacion** = `MAREABIO` / `MAREA` — cuanto mejora solo por las
  fases biologicas, con el ruido igual de inerte en ambos.
- **formulacion con las unidades ya arregladas** = `MAREABIO_REL` / `MAREA_REL`
  — **esta es la que vale para el paper.** Si aqui la razon se acerca a 1, la
  contribucion biologica se cae y lo que habia era un bug de escala.

Metrica: media de las 51 corridas, empates a 1e-8 (setup experimental, 5 y 6).


In [ ]:
import json
from scipy.stats import rankdata, wilcoxon

ARMAS   = ['MAREA', 'MAREA_REL', 'MAREABIO', 'MAREABIO_REL']
LIMPIOS = ['ABC', 'DE', 'GWO', 'PSO', 'WOA']   # 1.02x del presupuesto

def _load(alg, nd, mf):
    for d in (RESULTS_DIR, PROJECT_DIR, '.'):
        path = os.path.join(d, f'{alg}_D{nd}_FES{mf}.json')
        if os.path.exists(path):
            M = np.full((71, N_RUNS), np.nan)
            for k, v in json.load(open(path)).items():
                a, b = k.split('_')
                M[int(a[3:]), int(b[3:])] = v
            return M if np.isfinite(M).mean() > 0.99 else None
    return None

def _ranks(col, tol=1e-8):
    r = rankdata(col); o = np.argsort(col); out = r.copy(); i = 0
    while i < len(o):
        j = i
        while j + 1 < len(o) and abs(col[o[j+1]] - col[o[i]]) < tol:
            j += 1
        if j > i:
            a = np.mean([r[o[k]] for k in range(i, j+1)])
            for k in range(i, j+1):
                out[o[k]] = a
        i = j + 1
    return out

def _bew(X, Y):
    mx, my = X.mean(1), Y.mean(1)
    B = E = W = 0
    for q in range(71):
        try:    pv = wilcoxon(X[q], Y[q])[1]
        except Exception: pv = 1.0
        if pv >= 0.05:      E += 1
        elif mx[q] < my[q]: B += 1
        else:               W += 1
    return B, E, W

for mf in MAXFES:
    for nd in DIMS:
        arms = {a: _load(a, nd, mf) for a in ARMAS}
        arms = {a: m for a, m in arms.items() if m is not None}
        if len(arms) < 2:
            continue
        print(f'\n{"="*70}\nD={nd}   MaxFES={mf:,}   armas presentes: {", ".join(arms)}')

        def razon(a, b):
            if a not in arms or b not in arms:
                return None
            ma, mb = arms[a].mean(1), arms[b].mean(1)
            return np.nanmedian(ma / np.where(mb > 0, mb, np.nan))

        for etiqueta, a, b in (
                ('efecto escala        (MAREA_REL / MAREA)      ', 'MAREA_REL', 'MAREA'),
                ('efecto formulacion   (MAREABIO / MAREA)       ', 'MAREABIO', 'MAREA'),
                ('los dos juntos       (MAREABIO_REL / MAREA)   ', 'MAREABIO_REL', 'MAREA'),
                ('BIOLOGIA SOLA        (MAREABIO_REL/MAREA_REL) ', 'MAREABIO_REL', 'MAREA_REL')):
            r = razon(a, b)
            if r is None:
                print(f'  {etiqueta} (falta un arma)')
                continue
            B, E, W = _bew(arms[a], arms[b])
            print(f'  {etiqueta} razon={r:6.3f}   B/E/W={B:2d}/{E:2d}/{W:2d}')

        pool = dict(arms)
        for a in LIMPIOS:
            m = _load(a, nd, mf)
            if m is not None:
                pool[a] = m
        names = list(pool)
        means = np.array([pool[a].mean(1) for a in names])
        R  = np.column_stack([_ranks(means[:, q]) for q in range(71)])
        mr = R.mean(1); o = np.argsort(mr)
        print('  rank medio: ' + '  '.join(f'{names[i]}:{mr[i]:.2f}' for i in o))
